In [ ]:
from google.colab import files
uploaded = files.upload()

Saving matches.csv to matches.csv
Saving deliveries.csv to deliveries.csv


In [ ]:
import os
os.listdir()

['.config', 'matches.csv', 'deliveries.csv', 'sample_data']

In [ ]:
import pandas as pd

matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

matches.head()

,id,season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
0,1,2017,Hyderabad,05-04-2017,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
1,2,2017,Pune,06-04-2017,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant,field,normal,0,Rising Pune Supergiant,0,7,SPD Smith,Maharashtra Cricket Association Stadium,A Nand Kishore,S Ravi,NaN
2,3,2017,Rajkot,07-04-2017,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,normal,0,Kolkata Knight Riders,0,10,CA Lynn,Saurashtra Cricket Association Stadium,Nitin Menon,CK Nandan,NaN
3,4,2017,Indore,08-04-2017,Rising Pune Supergiant,Kings XI Punjab,Kings XI Punjab,field,normal,0,Kings XI Punjab,0,6,GJ Maxwell,Holkar Cricket Stadium,AK Chaudhary,C Shamshuddin,NaN
4,5,2017,Bangalore,08-04-2017,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,bat,normal,0,Royal Challengers Bangalore,15,0,KM Jadhav,M Chinnaswamy Stadium,NaN,NaN,NaN


In [ ]:
print(matches.columns)
print(deliveries.columns)

Index(['id', 'season', 'city', 'date', 'team1', 'team2', 'toss_winner',
       'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs',
       'win_by_wickets', 'player_of_match', 'venue', 'umpire1', 'umpire2',
       'umpire3'],
      dtype='object')
Index(['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball',
       'batsman', 'non_striker', 'bowler', 'is_super_over', 'wide_runs',
       'bye_runs', 'legbye_runs', 'noball_runs', 'penalty_runs',
       'batsman_runs', 'extra_runs', 'total_runs', 'player_dismissed',
       'dismissal_kind', 'fielder'],
      dtype='object')


In [ ]:
data = deliveries.merge(matches, left_on='match_id', right_on='id')

In [ ]:
data[['match_id', 'batting_team', 'bowling_team', 'total_runs', 'venue', 'winner']].head()

,match_id,batting_team,bowling_team,total_runs,venue,winner
0,1,Sunrisers Hyderabad,Royal Challengers Bangalore,0,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad
1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,0,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad
2,1,Sunrisers Hyderabad,Royal Challengers Bangalore,4,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad
3,1,Sunrisers Hyderabad,Royal Challengers Bangalore,0,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad
4,1,Sunrisers Hyderabad,Royal Challengers Bangalore,2,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad


In [ ]:
data_clean = data.copy()

In [ ]:
data_clean.isnull().sum()

,0
match_id,0
inning,0
batting_team,0
bowling_team,0
over,0
ball,0
batsman,0
non_striker,0
bowler,0
is_super_over,0


**Handling Missing values**


In [ ]:
data_clean['player_dismissed'].fillna("Not Out", inplace=True)
data_clean['dismissal_kind'].fillna("None", inplace=True)
data_clean['fielder'].fillna("None", inplace=True)

/tmp/ipykernel_2007/2673037407.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_clean['player_dismissed'].fillna("Not Out", inplace=True)
/tmp/ipykernel_2007/2673037407.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplac

In [ ]:
data_clean['city'].fillna("Unknown", inplace=True)

/tmp/ipykernel_2007/3906439909.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_clean['city'].fillna("Unknown", inplace=True)


In [ ]:
data_clean = data_clean[data_clean['winner'].notna()]

In [ ]:
data_clean.drop(columns=['umpire1', 'umpire2', 'umpire3'], inplace=True)

Summary - In player_dismissed, missing values were filled with "Not Out"
In dismissal_kind, missing values were filled with "None"
In fielder, missing values were filled with "None"
In city, missing values were filled with "Unknown"
Rows with missing values in winner were removed
The umpires column was dropped from the dataset


In [ ]:
data_clean['is_wicket'] = data_clean['player_dismissed'].apply(
    lambda x: 0 if x == "Not Out" else 1
)

**Match State Features**

In [ ]:
data_clean['current_score'] = data_clean.groupby('match_id')['total_runs'].cumsum()

In [ ]:
data_clean['wickets'] = data_clean.groupby('match_id')['is_wicket'].cumsum()

In [ ]:
data_clean['balls_bowled'] = (data_clean['over'] - 1) * 6 + data_clean['ball']

Summary - These features represent the live situation of the match at each ball:

current_score shows the total runs scored so far
wickets shows the number of wickets fallen so far
balls_bowled shows the progress of the match (how many balls have been bowled)

**FULL PIPELINE**

In [ ]:
second_innings = data_clean[data_clean['inning'] == 2].copy()

In [ ]:
first_innings_total = data_clean[data_clean['inning'] == 1] \
    .groupby('match_id')['total_runs'].sum().reset_index()

first_innings_total.rename(columns={'total_runs': 'target'}, inplace=True)

In [ ]:
second_innings = second_innings.merge(first_innings_total, on='match_id')

In [ ]:
second_innings['current_score'] = second_innings.groupby('match_id')['total_runs'].cumsum()

second_innings['runs_left'] = second_innings['target'] - second_innings['current_score']

second_innings['balls_bowled'] = (second_innings['over'] - 1) * 6 + second_innings['ball']

second_innings['balls_left'] = 120 - second_innings['balls_bowled']

second_innings['wickets'] = second_innings.groupby('match_id')['is_wicket'].cumsum()

second_innings['wickets_left'] = 10 - second_innings['wickets']

In [ ]:
second_innings['crr'] = (second_innings['current_score'] * 6) / second_innings['balls_bowled']

second_innings['rrr'] = (second_innings['runs_left'] * 6) / second_innings['balls_left']

In [ ]:
# Overs left
second_innings['overs_left'] = second_innings['balls_left'] / 6

# Pressure difference
second_innings['rrr_crr_diff'] = second_innings['rrr'] - second_innings['crr']

# Runs per wicket
second_innings['runs_per_wicket'] = second_innings['runs_left'] / (second_innings['wickets_left'] + 1)

In [ ]:
def get_phase(ball):
    if ball < 36:
        return 'powerplay'
    elif ball < 90:
        return 'middle'
    else:
        return 'death'

second_innings['phase'] = second_innings['balls_bowled'].apply(get_phase)

In [ ]:
second_innings['result'] = second_innings.apply(
    lambda row: 1 if row['batting_team'] == row['winner'] else 0,
    axis=1
)

In [ ]:
import numpy as np

# Remove invalid values
second_innings.replace([np.inf, -np.inf], np.nan, inplace=True)
second_innings.dropna(inplace=True)

# Remove noisy situations
second_innings = second_innings[
    (second_innings['balls_left'] > 12) &
    (second_innings['balls_left'] < 114) &
    (second_innings['runs_left'] > 0)
]

In [ ]:
features = ['batting_team', 'bowling_team',
            'runs_left', 'balls_left', 'wickets_left',
            'crr', 'rrr',
            'overs_left', 'rrr_crr_diff', 'runs_per_wicket',
            'phase']

Summary - This prepares IPL ball-by-ball data for win prediction modeling by focusing on the second innings (chasing phase). It calculates the target score from the first innings and merges it with second innings data. Key match state features such as runs left, balls left, wickets remaining, current run rate (CRR), and required run rate (RRR) are created. These features represent the real-time condition of a match at every ball. Finally, a target variable is generated to indicate whether the chasing team won or lost, making the dataset ready for machine learning.

**FIRST MODEL- Logistic Regression**

In [ ]:
from sklearn.model_selection import train_test_split

match_ids = second_innings['match_id'].unique()

train_ids, test_ids = train_test_split(match_ids, test_size=0.2, random_state=42)

train_data = second_innings[second_innings['match_id'].isin(train_ids)]
test_data = second_innings[second_innings['match_id'].isin(test_ids)]

In [ ]:
X_train = train_data[features]
y_train = train_data['result']

X_test = test_data[features]
y_test = test_data['result']

In [ ]:
import pandas as pd

X_train = pd.get_dummies(X_train, columns=['batting_team', 'bowling_team', 'phase'])
X_test = pd.get_dummies(X_test, columns=['batting_team', 'bowling_team', 'phase'])

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.7338335396039604


In [ ]:
lr_model.predict_proba(X_test_scaled[:5])

array([[0.08844487, 0.91155513],
       [0.04094497, 0.95905503],
       [0.04370662, 0.95629338],
       [0.03037033, 0.96962967],
       [0.03692443, 0.96307557]])

**Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

Random Forest Accuracy: 0.7175123762376238


**XGBoost**

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))

XGBoost Accuracy: 0.7203743811881188


In [ ]:
import pickle

pickle.dump(xgb_model, open('model.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))
pickle.dump(X_train.columns, open('columns.pkl', 'wb'))

In [ ]:
from google.colab import files

files.download('model.pkl')
files.download('columns.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>